# Comparación y selección de modelos de clasificación

Este notebook entrena varios modelos para el mismo problema: predecir si el ingreso anual supera USD 50K. Todos reciben los mismos datos y las mismas transformaciones, de modo que la comparación sea justa.

La selección se realiza con validación. El conjunto de prueba permanece sin consultar hasta haber elegido al ganador.

## 1. Preparación de datos

Primero se carga y valida el dataset. El target se convierte a `0` para `<=50K` y `1` para `>50K`. Se reserva 20% como test final; ese conjunto simula datos nuevos que el equipo no utiliza para escoger el modelo.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

from Pipeline_adult import calculate_metrics
from src.config import MODEL_DIR, OUTPUT_DIR, RANDOM_STATE, TEST_SIZE, VALIDATION_SIZE
from src.data import load_adult, split_features_target
from src.modeling import build_candidate_pipelines
from src.quality import run_quality_gates

df = load_adult()
run_quality_gates(df)
X, y = split_features_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train, y_train, test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE, stratify=y_train
)

print("Ajuste:", X_fit.shape)
print("Validación:", X_validation.shape)
print("Test final:", X_test.shape)


## 2. Modelos candidatos

Se comparan cuatro formas distintas de aprender patrones:

- **Regresión logística:** crea una separación basada en una combinación ponderada de las características. Es un modelo sencillo, rápido y una buena referencia inicial para clasificación.
- **Árbol de decisión:** construye preguntas sucesivas, por ejemplo si la edad o las horas superan cierto valor. Puede representar relaciones no lineales, pero un árbol sin límites puede memorizar los datos.
- **K vecinos más cercanos (KNN):** clasifica una persona observando las personas más parecidas del conjunto de entrenamiento. Necesita variables en escalas comparables y puede ser lento con muchos registros.
- **Random Forest:** combina 200 árboles y decide mediante sus resultados conjuntos. Suele ser más estable que un árbol individual.

Cada candidato incluye el mismo tratamiento de faltantes, escalado y codificación categórica.

In [ ]:
candidates = build_candidate_pipelines()
list(candidates)


## 3. Entrenamiento y comparación en validación

Cada modelo aprende con el conjunto de ajuste y se mide con validación. Se calculan varias métricas porque la clase `>50K` es minoritaria.

ROC-AUC se utiliza como criterio principal porque mide qué tan bien ordena el modelo los casos positivos por encima de los negativos considerando todos los umbrales. F1 se utiliza para desempatar.

In [ ]:
comparison = []

for name, candidate in candidates.items():
    candidate.fit(X_fit, y_fit)
    comparison.append({
        "modelo": name,
        **calculate_metrics(candidate, X_validation, y_validation),
    })

comparison_df = (
    pd.DataFrame(comparison)
      .sort_values(["roc_auc", "f1"], ascending=False)
      .reset_index(drop=True)
)
comparison_df


## 4. Elección del ganador

El primer modelo de la tabla tiene el ROC-AUC más alto en validación. Esta elección se realiza antes de mirar el desempeño en test, evitando escoger el modelo por casualidad a partir de los datos reservados para la evaluación final.

In [ ]:
selected_name = comparison_df.loc[0, "modelo"]
print("Modelo seleccionado:", selected_name)


## 5. Reentrenamiento y evaluación final

Una vez elegido el algoritmo, se crea nuevamente y se entrena con todo el 80% reservado para entrenamiento, incluyendo la parte que antes sirvió como validación. Solo entonces se evalúa sobre test.

Las métricas de test representan la estimación final del desempeño. No se utilizan para cambiar nuevamente de modelo.

In [ ]:
selected_pipeline = build_candidate_pipelines()[selected_name]
selected_pipeline.fit(X_train, y_train)

test_metrics = {
    "modelo": selected_name,
    **calculate_metrics(selected_pipeline, X_test, y_test),
}
pd.DataFrame([test_metrics])


## 6. Guardado de resultados

Se guarda la tabla completa de candidatos, las métricas finales y el pipeline ganador. El archivo del modelo contiene tanto la preparación de características como el clasificador, por lo que recibe directamente las columnas originales.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

comparison_df["seleccionado"] = comparison_df["modelo"].eq(selected_name)
comparison_df.to_csv(OUTPUT_DIR / "comparacion_modelos.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "metricas_clasificacion.csv", index=False)
joblib.dump(selected_pipeline, MODEL_DIR / "pipeline_adult_income.pkl", compress=3)

print("Comparación y modelo guardados correctamente")


## 7. Interpretación

El mejor algoritmo no se decide por ser el más complejo ni por obtener el accuracy más alto. Se selecciona mediante una regla establecida previamente y sobre datos de validación. Las diferencias entre precision y recall también muestran el tipo de error que favorece cada modelo, información necesaria para elegir posteriormente un umbral según el uso del sistema.